# 09 — Dual-stream VideoMAE V2 + Graph–Spatial–Temporal Transformer (demo 50 từ)

Notebook này train độc lập mô hình cải tiến trên **đúng 50 từ, đúng sample và custom signer-disjoint split 65/25/10** do notebook 07 tạo trước cell train. Nó không đọc checkpoint hay chờ `baseline_report.json` của baseline, nên có thể chạy song song với quá trình train notebook 07. Notebook tái sử dụng video và graph cache `[64, 75, 7]`; không extract lại video, frame hay keypoint. Đây là demo split để phát triển/slide, không phải official ASL Citizen benchmark.

Kiến trúc làm đúng theo tài liệu: nhánh RGB là `VideoMAE V2 [FROZEN] → RGB Transformer`; nhánh pose là `Graph Encoder → Spatial Transformer → Temporal Transformer`. Hai chuỗi token được ghép bằng **Multi-Head Cross-Attention hai chiều**, rồi feature fusion và classifier 50 lớp. Graph Encoder, Spatial Transformer, Temporal Transformer, Cross-Attention, RGB Transformer và classifier đều được train; chỉ VideoMAE V2 và dữ liệu pose/keypoint đã cache là cố định.

`RUN_TEST=False` để test tiếp tục được giữ kín. Chọn kiến trúc/checkpoint bằng validation top-1, Macro-F1 và validation loss; chỉ bật test một lần sau khi đã chốt mô hình.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-dual-stream-demo50'  # @param {type:'string'}
RESULTS_ROOT_STR = '/content/drive/.shortcut-targets-by-id/1-oYEcvJh4ylv_f4AKkJkCjs3FgzjDBFE/silent-signal-results/asl_citizen'  # @param {type:'string'}
BASELINE_RUN_NAME = 'videomaev2_rgb_transformer_demo50_split65_25_10_compact64_v1'  # @param {type:'string'}
DUAL_RUN_NAME = 'videomaev2_graph_spatial_temporal_crossattn_demo50_split65_25_10_compact64_v1'  # @param {type:'string'}
MODEL_ID = 'OpenGVLab/VideoMAEv2-Base'  # @param {type:'string'}
MODEL_REVISION = '0e826d7e85e39f9d951e331cd91c5c2d8142d385'  # @param {type:'string'}
CLASS_COUNT = 50  # @param {type:'integer'}
MAX_EPOCHS = 40  # @param {type:'integer'}
BATCH_SIZE = 2  # @param {type:'integer'}
MAX_TRAIN_BATCHES = 0  # @param {type:'integer'}
MAX_EVAL_BATCHES = 0  # @param {type:'integer'}
RGB_EMBEDDING_DIM = 64  # @param {type:'integer'}
RGB_LAYERS = 1  # @param {type:'integer'}
RGB_HEADS = 2  # @param {type:'integer'}
RGB_DROPOUT = 0.5  # @param {type:'number'}
POSE_EMBEDDING_DIM = 64  # @param {type:'integer'}
POSE_GRAPH_LAYERS = 1  # @param {type:'integer'}
POSE_SPATIAL_LAYERS = 1  # @param {type:'integer'}
POSE_TEMPORAL_LAYERS = 1  # @param {type:'integer'}
POSE_HEADS = 2  # @param {type:'integer'}
POSE_DROPOUT = 0.4  # @param {type:'number'}
FUSION_HEADS = 2  # @param {type:'integer'}
FUSION_DROPOUT = 0.4  # @param {type:'number'}
LEARNING_RATE = 0.0003  # @param {type:'number'}
WEIGHT_DECAY = 0.04  # @param {type:'number'}
LABEL_SMOOTHING = 0.15  # @param {type:'number'}
EARLY_STOPPING_PATIENCE = 4  # @param {type:'integer'}
EARLY_STOPPING_MIN_DELTA = 0.005  # @param {type:'number'}
OVERFIT_MONITOR_PATIENCE = 2  # @param {type:'integer'}
OVERFIT_MIN_EPOCH = 6  # @param {type:'integer'}
OVERFIT_LOSS_GAP = 0.5  # @param {type:'number'}
OVERFIT_TOP1_GAP = 0.2  # @param {type:'number'}
OVERFIT_VALIDATION_LOSS_REGRESSION = 0.1  # @param {type:'number'}
CHECKPOINT_EVERY = 20  # @param {type:'integer'}
PROGRESS_EVERY = 1  # @param {type:'integer'}
NUM_WORKERS = 2  # @param {type:'integer'}
DEVICE = 'auto'  # @param ['auto', 'cuda', 'cpu']
REQUIRE_CUDA = True  # @param {type:'boolean'}
RESUME = False  # @param {type:'boolean'}
RUN_TEST = False  # @param {type:'boolean'}
RUN_TRAINING = True  # @param {type:'boolean'}

PROJECT_ROOT = Path('/content/silent-signal')
RESULTS_ROOT = Path(RESULTS_ROOT_STR)
TOP200_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top200'
BASELINE_ROOT = TOP200_ROOT / 'baselines' / BASELINE_RUN_NAME
DATASET_ROOT = TOP200_ROOT / 'datasets/asl_citizen_top30'
GRAPH_ROOT = TOP200_ROOT / 'graph/asl_citizen_coco_wholebody_v1/t64'
GRAPH_REPORT = TOP200_ROOT / 'reports/graph_preparation_t64.json'
SELECTED_MANIFEST = BASELINE_ROOT / 'manifests/all.csv'
SELECTED_WORDS = BASELINE_ROOT / 'selected_50_words.json'
DUAL_ROOT = TOP200_ROOT / 'experiments' / DUAL_RUN_NAME
if CLASS_COUNT != 50: raise ValueError('Demo này được cố định ở đúng 50 từ.')

## Kiến trúc đúng theo tài liệu

```text
RGB video → VideoMAE V2 [FROZEN] → RGB Transformer ───────────────┐
                                                                  │
Whole-body keypoints cache → Graph Encoder → Spatial Transformer  │
                                      → Temporal Transformer ──────┤
                                                                  ↓
                                        Bidirectional Cross-Attention
                                                  ↓
                                            Feature Fusion
                                                  ↓
                                           Classifier 50 từ
```

Graph Encoder dùng adjacency giải phẫu đã biết để học quan hệ cục bộ; Spatial Transformer dùng self-attention giữa 75 joint trong cùng frame; Temporal Transformer dùng self-attention giữa 64 frame. Đây là ba khối riêng nối tiếp, không phải tên khác nhau của một khối.

In [ ]:
import os, subprocess, sys, time
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_JAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
def run(command):
    command = list(map(str, command))
    started = time.perf_counter()
    print(time.strftime('[%H:%M:%S] START'), ' '.join(command), flush=True)
    subprocess.run(command, check=True)
    print(time.strftime('[%H:%M:%S] DONE '), f'{time.perf_counter() - started:.1f}s', flush=True)
if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch', PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only', 'origin', PROJECT_GIT_REF])
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', PROJECT_ROOT])
run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.1.3', 'transformers==4.48.3', 'timm==1.0.15', 'easydict==1.13', 'opencv-python-headless==4.10.0.84', 'matplotlib==3.10.0'])
run([sys.executable, '-c', "import numpy; from transformers import PreTrainedModel; assert hasattr(numpy.dtypes, 'StringDType'); print('Import check PASS | NumPy', numpy.__version__)"])
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('CUDA=False: đang ở CPU runtime. Bấm Runtime > Change runtime type > T4 GPU, sau đó Disconnect and delete runtime, kết nối lại và chạy từ đầu. Không chạy train VideoMAE trên CPU.')

## Kiểm tra và ghép đúng artifact 50 từ

Cell này chỉ kiểm tra manifest/split, video và graph cache. Nó không cần checkpoint hay report kết quả của baseline 07; vì vậy notebook 09 có thể train song song ngay khi notebook 07 đã bước vào cell train. Nó cũng không đọc checkpoint smoke ở notebook 06 vì checkpoint đó chỉ chứng minh encoder cũ forward/backward được, không phải trọng số nghiên cứu đã train. Nhánh pose mới học từ đầu trên train split, nhưng dùng lại keypoint/graph cache đã extract.

In [ ]:
import csv, hashlib, json
required = [SELECTED_MANIFEST, SELECTED_WORDS, GRAPH_REPORT]
missing_artifacts = [str(path) for path in required if not path.is_file()]
if missing_artifacts: raise FileNotFoundError('Thiếu artifact từ notebook 05/07: ' + ', '.join(missing_artifacts))
with SELECTED_MANIFEST.open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
if len({row['sample_id'] for row in rows}) != len(rows): raise RuntimeError('sample_id bị trùng.')
split_ids = {split: {row['sample_id'] for row in rows if row['split'] == split} for split in ('train', 'validation', 'test')}
if split_ids['train'] & split_ids['validation'] or split_ids['train'] & split_ids['test'] or split_ids['validation'] & split_ids['test']:
    raise RuntimeError('LEAKAGE sample_id giữa custom split.')
split_signers = {split: {row['signer_id'] for row in rows if row['split'] == split} for split in ('train', 'validation', 'test')}
if split_signers['train'] & split_signers['validation'] or split_signers['train'] & split_signers['test'] or split_signers['validation'] & split_signers['test']:
    raise RuntimeError('LEAKAGE signer giữa custom split.')
def graph_path(sample_id):
    digest = hashlib.sha256(sample_id.encode('utf-8')).hexdigest()
    return GRAPH_ROOT / digest[:2] / f'{digest}.npz'
missing_videos = [row['video_path'] for row in rows if not (DATASET_ROOT / row['video_path']).is_file()]
missing_graphs = [row['sample_id'] for row in rows if not graph_path(row['sample_id']).is_file()]
if missing_videos: raise FileNotFoundError(f'Thiếu {len(missing_videos)} video; chạy xong notebook 07 trước.')
if missing_graphs: raise FileNotFoundError(f'Thiếu {len(missing_graphs)} graph cache; chạy xong notebook 05 trước.')
print('Paired artifact PASS:', {split: len(ids) for split, ids in split_ids.items()}, '| signers:', {split: len(ids) for split, ids in split_signers.items()})
print('Video cache:', DATASET_ROOT)
print('Graph cache:', GRAPH_ROOT)
print('Không cần xóa hoặc extract lại dữ liệu.')

## Train, resume và early stopping

Run này có thư mục riêng nên không ghi đè baseline. `last_checkpoint.pt` lưu tiến độ để resume khi Colab ngắt; `best_checkpoint.pt` lưu epoch có validation loss tốt nhất. Top-1 và Macro-F1 được ghi cho cả train/validation. `MAX_EPOCHS=40` chỉ là trần. Ngoài early stopping theo validation loss, bộ giám sát overfit ghi loss gap và top-1 gap mỗi epoch; nó chỉ dừng khi cả hai gap lớn **và** validation loss đã tụt khỏi best trong 2 epoch liên tiếp, nên không dừng chỉ vì train tốt hơn validation ở một epoch.

In [ ]:
def run_stream(command):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout: print(line, end='', flush=True)
    code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)
command = [sys.executable, '-u', '-m', 'silent_signal.cli.train_dual_stream_demo',
    '--manifest', SELECTED_MANIFEST, '--selection-report', SELECTED_WORDS,
    '--dataset-root', DATASET_ROOT,
    '--graph-root', GRAPH_ROOT, '--graph-report', GRAPH_REPORT, '--output-root', DUAL_ROOT,
    '--model-id', MODEL_ID, '--model-revision', MODEL_REVISION,
    '--classes', CLASS_COUNT, '--epochs', MAX_EPOCHS, '--batch-size', BATCH_SIZE,
    '--learning-rate', LEARNING_RATE, '--weight-decay', WEIGHT_DECAY,
    '--label-smoothing', LABEL_SMOOTHING, '--max-train-batches', MAX_TRAIN_BATCHES,
    '--max-eval-batches', MAX_EVAL_BATCHES, '--checkpoint-every', CHECKPOINT_EVERY,
    '--early-stopping-patience', EARLY_STOPPING_PATIENCE,
    '--early-stopping-min-delta', EARLY_STOPPING_MIN_DELTA,
    '--overfit-monitor-patience', OVERFIT_MONITOR_PATIENCE,
    '--overfit-min-epoch', OVERFIT_MIN_EPOCH, '--overfit-loss-gap', OVERFIT_LOSS_GAP,
    '--overfit-top1-gap', OVERFIT_TOP1_GAP,
    '--overfit-validation-loss-regression', OVERFIT_VALIDATION_LOSS_REGRESSION,
    '--progress-every', PROGRESS_EVERY, '--num-workers', NUM_WORKERS, '--device', DEVICE,
    '--rgb-embedding-dim', RGB_EMBEDDING_DIM, '--rgb-layers', RGB_LAYERS,
    '--rgb-heads', RGB_HEADS, '--rgb-dropout', RGB_DROPOUT,
    '--pose-embedding-dim', POSE_EMBEDDING_DIM, '--pose-graph-layers', POSE_GRAPH_LAYERS,
    '--pose-spatial-layers', POSE_SPATIAL_LAYERS, '--pose-temporal-layers', POSE_TEMPORAL_LAYERS,
    '--pose-heads', POSE_HEADS, '--pose-dropout', POSE_DROPOUT,
    '--fusion-heads', FUSION_HEADS, '--fusion-dropout', FUSION_DROPOUT]
if RESUME: command.append('--resume')
if RUN_TEST: command.append('--run-test')
if RUN_TRAINING: run_stream(command)
else: print('RUN_TRAINING=False — chỉ kiểm tra dữ liệu, chưa train.')

In [ ]:
from IPython.display import Image, display
report_path = DUAL_ROOT / 'dual_stream_report.json'
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    summary = {
        'best_epoch': report.get('best_epoch'),
        'trainable_parameters': report.get('trainable_parameters'),
        'validation': report.get('evaluation', {}).get('validation'),
        'early_stopping': report.get('early_stopping'),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    curve = DUAL_ROOT / 'training_curves.png'
    if curve.is_file(): display(Image(filename=str(curve)))
    print('PASS dual-stream validation demo. Chỉ bật RUN_TEST sau khi đã chốt cấu hình.')
else:
    print('Chưa có report; nếu runtime vừa ngắt, giữ RESUME=True và chạy lại:', DUAL_ROOT / 'last_checkpoint.pt')